# LoRA vs. LoRA+ on GPT-2 for SST-2

**Owner:** Manju Tadi

This notebook contains only the GPT-2 + SST-2 part of the team project.

## Experiment question

Does LoRA+ improve validation accuracy or convergence over standard LoRA on SST-2 while keeping the pretrained GPT-2 weights frozen?

## Planned comparison

- Standard LoRA: learning-rate ratio lambda = 1 (A and B use the same learning rate).
- LoRA+: test lambda = 4, 8, and 16 in short pilot runs; then repeat the selected setting with three seeds if compute allows.
- Shared settings: GPT-2, SST-2, max length 128, rank 8, alpha 8, AdamW, and the same data split and batch settings across methods.
- Primary metric: validation accuracy. Also record training time, peak GPU memory, trainable parameters, optimizer steps, and loss history.

This starter uses learning rate as 1e-4 for A and three epochs for the full run; keep these fixed across LoRA and LoRA+ comparisons. The 1-epoch pilot is only for checking the setup and comparing candidate ratios.


## 1. Environment and imports

Colab already includes PyTorch. Run the install line only if an import below fails; avoid upgrading packages in the middle of an active runtime.


In [ ]:
# If an import fails, uncomment and run this cell, then rerun the imports below.
%pip install -q transformers datasets peft evaluate accelerate

In [ ]:
import json
import time
from pathlib import Path

import evaluate
import numpy as np
import torch
import transformers
import datasets
import peft
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load and tokenize SST-2

We use the GLUE SST-2 train and validation splits. GPT-2 has no padding token by default, so EOS is reused for padding. The validation split is used for evaluation; the GLUE test labels are not used for local scoring.


In [ ]:
MODEL_NAME = "gpt2"
MAX_LENGTH = 128

raw_sst2 = load_dataset("nyu-mll/glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_sst2(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_sst2 = raw_sst2.map(tokenize_sst2, batched=True)
tokenized_sst2 = tokenized_sst2.remove_columns(["sentence", "idx"])
tokenized_sst2 = tokenized_sst2.rename_column("label", "labels")
tokenized_sst2.set_format("torch")

print(tokenized_sst2)
print("Train examples:", len(tokenized_sst2["train"]))
print("Validation examples:", len(tokenized_sst2["validation"]))
print("One tokenized example:", tokenized_sst2["train"][0])


## 3. Pilot and experiment settings

The pilot uses a small subset to check that the model can train and fit the available GPU. It is not the final benchmark. Set `PILOT_RUN = False` for the full SST-2 training set after the pilot is satisfactory.


In [ ]:
#run it only if the torchao compatibility error occurs.
#%pip uninstall -y torchao

In [ ]:
PILOT_RUN = False
PILOT_TRAIN_SIZE = 1000
PILOT_EVAL_SIZE = 200

train_dataset = tokenized_sst2["train"]
eval_dataset = tokenized_sst2["validation"]
if PILOT_RUN:
    train_dataset = train_dataset.select(range(min(PILOT_TRAIN_SIZE, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(PILOT_EVAL_SIZE, len(eval_dataset))))

# Proposal settings and a conservative T4 batch arrangement.
RANK = 8
LORA_ALPHA = 8
LORA_DROPOUT = 0.1
BASE_LR_A = 1e-4  # Starting value copied from the team's existing notebook.
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size 16.
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 1 if PILOT_RUN else 3
SEED = 42

# Run one configuration at a time. No training starts unless this is True.
RUN_TRAINING = False
EXPERIMENT_MODE = "standard_lora"  # "standard_lora" or "loraplus"
LAMBDA_RATIO = 1  # For LoRA+, change to 4, 8, or 16 for separate pilot runs.

# Checkpoints are off by default to conserve Drive storage.
SAVE_CHECKPOINTS = False
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/LoRA_Plus_Project/Manju_GPT2_SST2")
else:
    OUTPUT_ROOT = Path("/content/Manju_GPT2_SST2")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Mode:", "pilot" if PILOT_RUN else "full")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))
print("Epochs:", NUM_EPOCHS)
print("Run training:", RUN_TRAINING)


## 4. Build a fresh GPT-2 classifier and optimizer

The paper places GPT-2 LoRA adapters on `c_attn`, `c_proj`, and `c_fc`. The proposal uses rank 8 and alpha 8. The classification head (`score`) is trainable along with the adapters. Build a fresh model for each method, learning-rate ratio, and seed so one run cannot carry trained weights into another.

Standard LoRA uses the same learning rate for A, B, and the classification head. LoRA+ uses `learning_rate_B = lambda * learning_rate_A`; the classification head stays at the base learning rate.


In [ ]:
def build_gpt2_lora_model(seed):
    set_seed(seed)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["c_attn", "c_proj", "c_fc"],
        modules_to_save=["score"],
        bias="none",
        fan_in_fan_out=True,
    )
    return get_peft_model(base_model, lora_config)

def build_optimizer(model, mode, lambda_ratio):
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise ValueError("No trainable parameters found.")

    if mode == "standard_lora":
        return torch.optim.AdamW(
            trainable,
            lr=BASE_LR_A,
            weight_decay=WEIGHT_DECAY,
        )

    if mode != "loraplus":
        raise ValueError("EXPERIMENT_MODE must be 'standard_lora' or 'loraplus'.")
    if lambda_ratio not in (4, 8, 16):
        raise ValueError("For this project, LoRA+ pilot lambda values are 4, 8, or 16.")

    lora_a, lora_b, other = [], [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if "lora_A" in name:
            lora_a.append(parameter)
        elif "lora_B" in name:
            lora_b.append(parameter)
        else:
            other.append(parameter)

    if not lora_a or not lora_b:
        raise ValueError("Could not find both LoRA A and B trainable parameters.")

    groups = [
        {"params": lora_a, "lr": BASE_LR_A},
        {"params": lora_b, "lr": BASE_LR_A * lambda_ratio},
    ]
    if other:
        groups.append({"params": other, "lr": BASE_LR_A})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


## 5. Metrics and training

SST-2 is scored with accuracy. This cell runs one selected configuration only; it does not automatically launch all LoRA+ ratios or all seeds. Keep `RUN_TRAINING = False` while checking setup. Set it to `True` only when you are ready to run the selected pilot.


In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

if RUN_TRAINING:
    if EXPERIMENT_MODE not in ("standard_lora", "loraplus"):
        raise ValueError("Choose 'standard_lora' or 'loraplus'.")

    model = build_gpt2_lora_model(SEED)
    model.print_trainable_parameters()
    trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_parameter_count = sum(p.numel() for p in model.parameters())
    optimizer = build_optimizer(model, EXPERIMENT_MODE, LAMBDA_RATIO)

    ratio_label = 1 if EXPERIMENT_MODE == "standard_lora" else LAMBDA_RATIO
    run_name = f"{EXPERIMENT_MODE}_lambda{ratio_label}_seed{SEED}_{'pilot' if PILOT_RUN else 'full'}"
    run_output_dir = OUTPUT_ROOT / run_name

    training_args = TrainingArguments(
        output_dir=str(run_output_dir),
        eval_strategy="epoch",
        save_strategy="epoch" if SAVE_CHECKPOINTS else "no",
        save_total_limit=1,
        logging_strategy="steps",
        logging_steps=50,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        fp16=torch.cuda.is_available(),
        report_to="none",
        load_best_model_at_end=SAVE_CHECKPOINTS,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_accuracy,
        optimizers=(optimizer, None),
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    train_output = trainer.train()
    elapsed_seconds = time.time() - start_time
    final_eval = trainer.evaluate()
    peak_gpu_memory_mb = (torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else None

    result = {
        "model": MODEL_NAME,
        "task": "GLUE SST-2",
        "method": EXPERIMENT_MODE,
        "lambda_ratio": ratio_label,
        "seed": SEED,
        "pilot_run": PILOT_RUN,
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "epochs": NUM_EPOCHS,
        "validation_accuracy": final_eval.get("eval_accuracy"),
        "training_time_seconds": elapsed_seconds,
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "trainable_parameters": trainable_parameter_count,
        "total_parameters": total_parameter_count,
        "optimizer_steps": trainer.state.global_step,
        "train_loss": train_output.metrics.get("train_loss"),
        "evaluation_history": [
            {k: log[k] for k in ("epoch", "eval_loss", "eval_accuracy") if k in log}
            for log in trainer.state.log_history if "eval_accuracy" in log
        ],
    }
    print(json.dumps(result, indent=2))
    with open(run_output_dir / "metrics.json", "w") as f:
        json.dump(result, f, indent=2)
else:
    print("Training did not start. Set RUN_TRAINING = True when ready to run one selected configuration.")


## 6. Run plan

1. Run the setup, data, configuration, and helper cells. Confirm the train and validation counts and the tokenized example.
2. Pilot standard LoRA with `EXPERIMENT_MODE = "standard_lora"`, `RUN_TRAINING = True`, and the small pilot enabled.
3. Pilot LoRA+ once each with `EXPERIMENT_MODE = "loraplus"` and `LAMBDA_RATIO = 4`, `8`, and `16`. Use the same seed, subset, and settings for these comparison runs.
4. Select the best pilot setting based on validation accuracy and loss/convergence behavior. Turn off pilot mode and repeat that LoRA+ setting **and the standard LoRA baseline** with three seeds each, if compute allows.
5. Preserve each printed `metrics.json` result. Combine your GPT-2/SST-2 results with Ram's and Asneem's results in the team's final table later.

Do not use the GLUE test split for local scoring; its labels are not provided. The proposal's primary SST-2 metric is validation accuracy. Paper-exact extra runs, such as the paper's longer SST-2 schedule or wider learning-rate grid, can wait until the proposal's core experiments are complete.


Diognestic Cell

In [ ]:
if RUN_TRAINING:
    train_check = trainer.evaluate(
        eval_dataset=train_dataset,
        metric_key_prefix="train",
    )
    print(train_check)

    val_output = trainer.predict(eval_dataset)
    val_logits = val_output.predictions
    if isinstance(val_logits, tuple):
        val_logits = val_logits[0]

    val_predictions = np.argmax(val_logits, axis=-1)
    print("Validation predicted label counts:",
        np.bincount(val_predictions, minlength=2))
    print("Validation true label counts:",
        np.bincount(np.asarray(eval_dataset["labels"]), minlength=2))

In [ ]:
if RUN_TRAINING:    
    from google.colab import drive
    drive.mount("/content/drive")

    import csv
    import json
    import shutil
    from pathlib import Path

    # Your Drive folder for this project
    drive_dir = Path("/content/drive/MyDrive/LoRA_Plus_Project/Manju_GPT2_SST2")
    drive_dir.mkdir(parents=True, exist_ok=True)

    # Read the metrics saved by the completed run
    source_json = run_output_dir / "metrics.json"
    with open(source_json) as f:
        saved_metrics = json.load(f)

    # Keep a copy of the detailed metrics and loss history
    run_name = run_output_dir.name
    run_dir_on_drive = drive_dir / run_name
    run_dir_on_drive.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_json, run_dir_on_drive / "metrics.json")

    # One summary row for the CSV
    row = {
        "run_name": run_name,
        "model": saved_metrics["model"],
        "task": saved_metrics["task"],
        "method": saved_metrics["method"],
        "lambda_ratio": saved_metrics["lambda_ratio"],
        "seed": saved_metrics["seed"],
        "pilot_run": saved_metrics["pilot_run"],
        "train_examples": saved_metrics["train_examples"],
        "eval_examples": saved_metrics["eval_examples"],
        "epochs": saved_metrics["epochs"],
        "train_accuracy": train_check.get("train_accuracy", ""),
        "validation_accuracy": saved_metrics.get("validation_accuracy", ""),
        "train_loss": saved_metrics.get("train_loss", ""),
        "validation_loss": final_eval.get("eval_loss", ""),
        "training_time_seconds": saved_metrics.get("training_time_seconds", ""),
        "peak_gpu_memory_mb": saved_metrics.get("peak_gpu_memory_mb", ""),
        "trainable_parameters": saved_metrics.get("trainable_parameters", ""),
        "total_parameters": saved_metrics.get("total_parameters", ""),
        "optimizer_steps": saved_metrics.get("optimizer_steps", ""),
    }

    # Add or replace this run's row in the master CSV
    csv_path = drive_dir / "results.csv"
    fieldnames = list(row.keys())
    existing_rows = []

    if csv_path.exists():
        with open(csv_path, newline="") as f:
            existing_rows = list(csv.DictReader(f))

    existing_rows = [r for r in existing_rows if r["run_name"] != run_name]
    existing_rows.append(row)

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(existing_rows)

    print("Saved run metrics to:", run_dir_on_drive)
    print("Updated results table:", csv_path)